# Persian Sentiment Lexicon Builder
ساخت sentiment lexicon فارسی با استفاده از ParsBERT

In [ ]:
# ─────────────────────────────────────────
# Section 1: Install & Import
# ─────────────────────────────────────────
!pip install hazm transformers torch -q

import hazm
import torch
import json
import numpy as np
import pandas as pd
from collections import Counter
from transformers import AutoTokenizer, AutoModelForMaskedLM

print('✓ Imports done')

: 

In [ ]:
# ─────────────────────────────────────────
# Section 2: Config
# ─────────────────────────────────────────

# ── Data ──────────────────────────────────
CSV_PATH       = 'clean_comments.csv' 
TEXT_COL       = 'comment'
RATING_COL     = 'rating'
MAX_COMMENTS   = 500

# ── کلمات کاندید ──────────────────────────
MIN_WORD_FREQ  = 0                # حداقل تکرار کلمه در corpus
MIN_WORD_LEN   = 1                # حداقل طول کلمه
MAX_CANDIDATES = 500             # None = همه کاندیدها

# ── مدل و inference ───────────────────────
MODEL_NAME     = 'HooshvareLab/bert-base-parsbert-uncased'
BATCH_SIZE     = 32
TOP_K          = 50               # تعداد پیش‌بینی‌های برتر BERT

# ── POS tagهای مجاز ───────────────────────
VALID_POS      = {'N', 'AJ', 'ADV'}  # اسم، صفت، قید

POSITIVE_SEEDS = {
    'راضی', 'خوشحال', 'ممنون', 'خوب', 'مناسب',
    'خوشحالم', 'راضیم', 'ممنونم'
}

NEGATIVE_SEEDS = {
    'پشیمان', 'ناراحت', 'ناراضی', 'خسته', 'عصبانی',
    'شوکه', 'اذیت', 'منصرف', 'پشیمانم', 'ناراحتم',
    'ناراضیم', 'خسته‌ام', 'عصبانیم'
}

# ── template جمله‌ها ───────────────────────
# {word} = کلمه هدف که اینجا قرار می‌گیرد
# [MASK] = جایی که BERT پر می‌کند
POSITIVE_TEMPLATES = [
    'محصول {word} بود و از خریدم [MASK] هستم',      
]

NEGATIVE_TEMPLATES = [
    'محصول {word} بود و از خریدم [MASK] شدم',        
]

print('✓ Config loaded')

In [ ]:
# ─────────────────────────────────────────
# Section 3: Load Model & hazm
# ─────────────────────────────────────────
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

normalizer = hazm.Normalizer()
stemmer    = hazm.Stemmer()
tagger = hazm.POSTagger(
    repo_id="roshan-research/hazm-postagger",
    model_filename="pos_tagger.model"
)
print(f'✓ Model loaded | device: {device}')

In [ ]:
# ─────────────────────────────────────────
# Section 4: Load & Limit Comments
# ─────────────────────────────────────────
df = pd.read_csv(CSV_PATH).dropna(subset=[TEXT_COL])

if MAX_COMMENTS:
    df = df.head(MAX_COMMENTS)

comments = df[TEXT_COL].tolist()
ratings  = df[RATING_COL].tolist() if RATING_COL in df.columns else []

print(f'✓ Loaded {len(comments)} comments')
df[[TEXT_COL, RATING_COL]].head(3)

In [ ]:
# Section 5: Extract Candidate Words (fixed)
from tqdm import tqdm

VALID_POS_PREFIX = {'NOUN', 'ADJ', 'ADV'}

def extract_candidates(comments):
    word_freq      = Counter()
    total_words    = 0
    filtered_pos   = 0
    filtered_len   = 0

    with tqdm(total=len(comments), desc='Extracting words', unit='comment') as pbar:
        for i, comment in enumerate(comments):
            normalized = normalizer.normalize(comment)
            sentences  = hazm.sent_tokenize(normalized)

            for sent in sentences:
                words  = hazm.word_tokenize(sent)
                tagged = tagger.tag(words)

                for word, pos in tagged:
                    total_words += 1
                    base_pos = pos.split(',')[0]

                    if base_pos not in VALID_POS_PREFIX:
                        filtered_pos += 1
                        continue
                    if len(word) < MIN_WORD_LEN:
                        filtered_len += 1
                        continue

                    word_freq[word] += 1

            # آپدیت postfix هر ۱۰ کامنت
            if (i + 1) % 10 == 0:
                pbar.set_postfix({
                    'unique': len(word_freq),
                    'total':  total_words,
                    'fil_pos': filtered_pos,
                    'fil_len': filtered_len,
                })
            pbar.update(1)

    candidates = [w for w, freq in word_freq.items()
                  if freq >= MIN_WORD_FREQ]

    print(f'\n── Extract summary ──────────────────')
    print(f'  کل توکن‌های پردازش شده : {total_words:,}')
    print(f'  فیلتر شده (POS)        : {filtered_pos:,}')
    print(f'  فیلتر شده (طول)        : {filtered_len:,}')
    print(f'  unique words یافت شده  : {len(word_freq):,}')
    print(f'  بعد از فیلتر فرکانس    : {len(candidates):,}')
    print(f'────────────────────────────────────')

    return candidates, word_freq


candidates, word_freq = extract_candidates(comments)

if MAX_CANDIDATES:
    candidates = candidates[:MAX_CANDIDATES]

print(f'\n✓ {len(candidates)} candidates ready for scoring')
print('Top 20:', word_freq.most_common(20))

In [ ]:
# ─────────────────────────────────────────
# Section 6: Batch MLM Scoring
# ─────────────────────────────────────────
def get_mask_predictions_batch(sentences):
    """یک batch از جملات را می‌گیرد، هر کدام باید یک [MASK] داشته باشند"""
    inputs = tokenizer(
        sentences,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=128
    ).to(device)

    mask_token_id = tokenizer.mask_token_id
    mask_indices  = (inputs['input_ids'] == mask_token_id).nonzero(as_tuple=False)

    with torch.no_grad():
        logits = model(**inputs).logits

    probs   = torch.softmax(logits, dim=-1)
    results = []

    for i in range(len(sentences)):
        row_mask = mask_indices[mask_indices[:, 0] == i]
        if len(row_mask) == 0:
            results.append({})
            continue

        mask_pos         = row_mask[0, 1]
        top_probs, top_ids = torch.topk(probs[i, mask_pos, :], TOP_K)

        preds = {tokenizer.decode([idx]).strip(): prob.item()
                 for prob, idx in zip(top_probs, top_ids)}
        results.append(preds)

    return results


def build_sentences(candidates):
    """همه جملات را برای همه کاندیدها می‌سازد"""
    sentences, meta = [], []
    for word in candidates:
        for i, (t_pos, t_neg) in enumerate(zip(POSITIVE_TEMPLATES,
                                               NEGATIVE_TEMPLATES)):
            sentences.append(t_pos.replace('{word}', word))
            meta.append((word, i, 'pos'))

            sentences.append(t_neg.replace('{word}', word))
            meta.append((word, i, 'neg'))
    return sentences, meta


POSITIVE_TEMPLATES = POSITIVE_TEMPLATES[:1]
NEGATIVE_TEMPLATES = NEGATIVE_TEMPLATES[:1]

def score_all_batch(candidates):
    sentences, meta = build_sentences(candidates)
    all_preds        = []
    total_batches    = (len(sentences) + BATCH_SIZE - 1) // BATCH_SIZE

    print(f'Total sentences: {len(sentences)} | Batches: {total_batches}\n')

    for b in range(total_batches):
        start = b * BATCH_SIZE
        end   = min(start + BATCH_SIZE, len(sentences))
        preds = get_mask_predictions_batch(sentences[start:end])
        all_preds.extend(preds)
        if (b + 1) % 20 == 0 or (b + 1) == total_batches:
            print(f'  batch [{b+1}/{total_batches}] done')

    # جمع‌آوری جداگانه pos و neg با counter مستقل
    acc = {
        w: {'pos': 0.0, 'neg': 0.0, 'pos_n': 0, 'neg_n': 0}
        for w in candidates
    }

    for preds, (word, idx, polarity) in zip(all_preds, meta):
        if polarity == 'pos':
            s = sum(p for w, p in preds.items() if w in POSITIVE_SEEDS)
            acc[word]['pos']   += s
            acc[word]['pos_n'] += 1
        else:
            s = sum(p for w, p in preds.items() if w in NEGATIVE_SEEDS)
            acc[word]['neg']   += s
            acc[word]['neg_n'] += 1

    lexicon = {}
    for word, s in acc.items():
        pos_mean = s['pos'] / max(s['pos_n'], 1)
        neg_mean = s['neg'] / max(s['neg_n'], 1)
        lexicon[word] = round(pos_mean - neg_mean, 4)

    return lexicon

print('✓ Scoring functions ready')

In [ ]:
# ─────────────────────────────────────────
# Section 7: Run & Build Lexicon
# ─────────────────────────────────────────
lexicon = score_all_batch(candidates)
print(f'\n✓ Lexicon built: {len(lexicon)} words')

In [ ]:
# Section 8 — save را عوض کن

POSITIVE_THRESHOLD =  0.5
NEGATIVE_THRESHOLD = -0.05

def label(score):
    if score >  POSITIVE_THRESHOLD: return 'positive'
    if score <  NEGATIVE_THRESHOLD: return 'negative'
    return 'neutral'

rich_lexicon = {
    word: {
        'score': score,
        'label': label(score)
    }
    for word, score in lexicon.items()
}

with open('persian_sentiment_lexicon.json', 'w', encoding='utf-8') as f:
    json.dump(rich_lexicon, f, ensure_ascii=False, indent=2)

# نمایش تفکیک شده
sorted_lex = sorted(lexicon.items(), key=lambda x: x[1], reverse=True)
positives  = [(w,s) for w,s in sorted_lex if s >  POSITIVE_THRESHOLD]
negatives  = [(w,s) for w,s in sorted_lex if s <  NEGATIVE_THRESHOLD]
neutrals   = [(w,s) for w,s in sorted_lex if NEGATIVE_THRESHOLD <= s <= POSITIVE_THRESHOLD]

print(f'=== Positive: {len(positives)} words ===')
for w,s in positives[:10]: print(f'  {w}: {s:+.4f}')

print(f'\n=== Negative: {len(negatives)} words ===')
for w,s in negatives[:10]: print(f'  {w}: {s:+.4f}')

print(f'\n=== Neutral: {len(neutrals)} words ===')
for w,s in neutrals[:10]: print(f'  {w}: {s:+.4f}')

print(f'\n✓ Saved — {len(positives)} pos / {len(negatives)} neg / {len(neutrals)} neutral')